In [115]:
import pandas as pd
import tqdm

In [116]:
train = pd.read_csv("data_train.csv")

In [117]:
train

,loc_x,loc_y,minutes_remaining,shot_distance,shot_made_flag,shot_id
0,0,0,7,0,1,1
1,-3,130,9,13,0,2
2,82,8,7,8,0,3
3,100,101,10,14,1,4
4,0,0,9,0,1,5
...,...,...,...,...,...,...
24995,97,43,1,10,0,24996
24996,18,77,2,7,0,24997
24997,234,4,0,23,0,24998
24998,75,20,2,7,0,24999


In [118]:
train.describe()

,loc_x,loc_y,minutes_remaining,shot_distance,shot_made_flag,shot_id
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.00000,25000.000000
mean,7.125400,91.287880,4.885120,13.458520,0.44744,12500.500000
std,110.029921,88.289321,3.452593,9.397722,0.49724,7217.022701
min,-250.000000,-44.000000,0.000000,0.000000,0.00000,1.000000
25%,-67.250000,4.000000,2.000000,5.000000,0.00000,6250.750000
50%,0.000000,72.000000,5.000000,15.000000,0.00000,12500.500000
75%,94.000000,160.000000,8.000000,21.000000,1.00000,18750.250000
max,248.000000,791.000000,11.000000,79.000000,1.00000,25000.000000


In [119]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        #######Please design your model here########
        self.hidden1 = nn.Linear(2, 6)
        self.ac1 = nn.ReLU()
        self.hidden2 = nn.Linear(6, 6)
        self.ac2 = nn.ReLU()
        self.output = nn.Linear(6, 1)
        self.ac3 = nn.Sigmoid()
        
    def forward(self, x):
        #######Please design your model here########
        x = self.ac1(self.hidden1(x))
        x = self.ac2(self.hidden2(x))
        x = self.ac3(self.output(x))
        return x



In [120]:
model = MyModel()

loss_fn = nn.MSELoss()
batch_size = 32

In [121]:
opt = torch.optim.Adam(model.parameters(), lr=5e-4)


In [122]:
train_X = train[["loc_x", "loc_y"]].values
train_y = train["shot_made_flag"].values

train_X_tens = torch.Tensor(train_X)
train_y_tens = torch.Tensor(train_y).reshape(-1, 1)

In [123]:
from sklearn.model_selection import train_test_split

train_X_tens, test_X_tens, train_y_tens, test_y_tens = train_test_split(train_X_tens, train_y_tens, test_size=0.1)

In [124]:
import copy

In [125]:
import numpy as np

best_model = model
prev_acc = 0
cnt = 0

def get_acc(test_X, test_y, model):
    model.eval()
    out = model(test_X).detach().numpy()
    # print(out)
    pred = [1 if x >= 0.5 else 0 for x in out]

    # print(out)

    acc = (np.array(pred).reshape(-1) == test_y.detach().numpy().reshape(-1)).sum() / len(out)
    return acc

def train_(epoch):
    global prev_acc, best_model, cnt
    print("Epoch: ", epoch)

    loss_curr = 0;
    for i in (pt := tqdm.tqdm(range(0, len(train_X), batch_size))):
        X = train_X_tens[i:i + batch_size]
        y = train_y_tens[i:i + batch_size]


        output = model(X)
        loss = loss_fn(output, y)
        loss_curr += (loss.item())


        pt.set_postfix({"Loss": loss_curr})
        opt.zero_grad()
        loss.backward()
        opt.step()
    

    with torch.no_grad():
        acc = (get_acc(test_X_tens, test_y_tens, model=model))
        print(acc)

    if(acc > prev_acc):
        prev_acc = acc
        best_model = copy.deepcopy(model)
    


In [126]:
for i in range(200):
    if(train_(i) == 0): break
    

Epoch:  0


100%|██████████| 782/782 [00:02<00:00, 315.71it/s, Loss=nan] 


0.5364
Epoch:  1


100%|██████████| 782/782 [00:02<00:00, 340.01it/s, Loss=nan] 


0.5444
Epoch:  2


...

0.6008
Epoch:  198


100%|██████████| 782/782 [00:02<00:00, 339.53it/s, Loss=nan] 


0.6008
Epoch:  199


100%|██████████| 782/782 [00:02<00:00, 340.64it/s, Loss=nan] 


0.6008


In [127]:
acc = (get_acc(test_X_tens, test_y_tens, model=best_model))


In [128]:
acc

0.6052

In [132]:
model_code = """  
import torch
import torch.nn as nn


class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        #######Please design your model here########
        self.hidden1 = nn.Linear(2, 6)
        self.ac1 = nn.ReLU()
        self.hidden2 = nn.Linear(6, 6)
        self.ac2 = nn.ReLU()
        self.output = nn.Linear(6, 1)
        self.ac3 = nn.Sigmoid()
        
    def forward(self, x):
        #######Please design your model here########
        x = self.ac1(self.hidden1(x))
        x = self.ac2(self.hidden2(x))
        x = self.ac3(self.output(x))
        return x


"""
# Save the model structure
with open('submission_model.py', 'w') as f:
    f.write(model_code)
print("submission_model.py file is generated.")

submission_model.py file is generated.


In [133]:
torch.save(best_model.state_dict(), 'submission_dic.pth')

In [134]:
import zipfile
import os

# Define the files to be packaged and the compressed file name. 
files_to_zip = ['submission_model.py', 'submission_dic.pth']
zip_filename = 'submission.zip'

# Create a zip file to submit.
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Add files to the zip file
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created successfully!')

submission.zip is created successfully!
